# 08d -- DraftSharks Rankings

**Purpose**: Scrape DraftSharks 2026 dynasty rookie rankings and ingest into
`fact_rookie_rankings`. Self-contained — run independently of other source notebooks.

**Source**: Single page — ordered `<ol><li>` list, one player per item.

| Field | Notes |
|---|---|
| `global_rank` | List position (1-based) |
| `positional_rank` | Computed from list order within each position group |
| `grade` | Not published — stored as `null` |
| `school_raw` | Not published in list format — stored as `""` |
| `rank_date` | Parsed from "Updated [Month DD, YYYY]" in page header |

**Note**: Free-tier page returns top 90 players; final 10 are behind a paywall.

**Prerequisites**: `08_fact_rookie_rankings_seed.ipynb`, `01_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended), `data/review_fuzzy_matches.csv` (if needed)

## 1 -- Setup & Shared Helpers

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import requests
import re
import json
import time
from pathlib import Path
from datetime import date, datetime
from dataclasses import dataclass, field
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from thefuzz import fuzz, process

# ---- Shared helpers + config from etl_helpers -------------------------------
import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (
    LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS, DEFAULT_HEADERS,
    _make_session, _parse_rank_date, clean_player_name, generate_player_key,
    add_players_from_source, ingest_ranking_source, append_review,
)


In [2]:
# clean_player_name / generate_player_key / _parse_rank_date imported from etl_helpers

Helpers OK


## 2 -- add_players_from_source

In [ ]:
# add_players_from_source imported from etl_helpers (canonical, alias-aware)

## 3 -- ingest_ranking_source

In [ ]:
# ingest_ranking_source imported from etl_helpers

## 4 -- DraftSharksScraper

Parses the first `<ol>` on the page — each `<li>` is one player in ranked order.

**Item format**: `"Name, POS, NFL Team"` (comma-separated, 3 fields).

**`positional_rank`**: derived by counting occurrences of each `position_group`
in list order — not provided by DraftSharks directly.

**`rank_date`**: extracted from "Updated [Month DD, YYYY]" in the page header.

In [5]:
class DraftSharksScraper:
    # Parses the ranked <ol> list from DraftSharks 2026 dynasty rookie rankings.
    # Item format: "Name, POS, NFL Team" — positional_rank computed from list order.
    # Returns up to 90 players (final 10 are behind a paywall on the free tier).

    _URL = "https://www.draftsharks.com/article/2026-rookie-rankings"
    _DATE_PAT = re.compile(
        r"(?:Updated\s+on\s+|Updated\s+)([A-Za-z]+\s+\d{1,2},?\s*\d{4})",
        re.IGNORECASE,
    )

    _HEADERS = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    SOURCES = {
        "DraftSharks": {
            "site_reference": "2026 Dynasty Rookie Rankings: Top 100",
            "source_site":    "DraftSharks",
        },
    }

    def __init__(self, timeout: int = 30, delay: float = 1.5):
        self.timeout = timeout
        self.delay   = delay
        self.session = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)

    def scrape(self, source_key: str = "DraftSharks") -> tuple[pd.DataFrame, str | None]:
        time.sleep(self.delay)
        resp = self.session.get(self._URL, timeout=self.timeout)
        resp.raise_for_status()

        from bs4 import BeautifulSoup
        soup = BeautifulSoup(resp.text, "html.parser")

        # rank_date — prefer article:modified_time meta tag (full ISO datetime)
        # Falls back to _DATE_PAT regex on page text if meta tag is absent.
        meta_date = (
            soup.find("meta", property="article:modified_time") or
            soup.find("meta", property="article:published_time")
        )
        if meta_date and meta_date.get("content"):
            rank_date = _parse_rank_date(meta_date["content"][:10])  # YYYY-MM-DD prefix
        else:
            m_date = self._DATE_PAT.search(resp.text)
            rank_date = _parse_rank_date(m_date.group(1) if m_date else None)

        # First <ol> is the rankings list
        ol = soup.find("ol")
        if not ol:
            raise ValueError("No <ol> found on DraftSharks page")

        # Track positional_rank per position_group
        pos_map    = pd.read_parquet(DATA / "dim_position.parquet")[["position_raw", "position_group"]]
        pos_lookup = dict(zip(pos_map["position_raw"], pos_map["position_group"]))
        pos_counts: dict[str, int] = {}

        rows = []
        for global_rank, li in enumerate(ol.find_all("li"), start=1):
            text  = li.get_text(separator=",", strip=True)
            parts = [p.strip() for p in text.split(",")]
            if len(parts) < 2:
                continue
            player_name  = parts[0]
            position_raw = parts[1].upper()

            # Map position_raw -> position_group for positional_rank tracking
            pos_group = pos_lookup.get(position_raw, position_raw)
            pos_counts[pos_group] = pos_counts.get(pos_group, 0) + 1
            pos_rank = pos_counts[pos_group]

            rows.append({
                "player_name":     player_name,
                "position_raw":    position_raw,
                "school_raw":      "",       # not available in DraftSharks list format
                "global_rank":     global_rank,
                "positional_rank": pos_rank,
                "grade":           None,     # DraftSharks does not publish grades
                "nfl_team":        parts[2] if len(parts) > 2 else "",
            })

        if not rows:
            raise ValueError("No player rows parsed from DraftSharks page")

        df = pd.DataFrame(rows)
        cfg = self.SOURCES[source_key]
        print(
            f"Scraped {source_key}: {len(df)} players | "
            f"{cfg['site_reference']} | rank_date={rank_date}"
        )
        return df, rank_date

## 5 -- Run DraftSharks Ingestion

In [6]:
SOURCES = DraftSharksScraper.SOURCES   # alias for validation cell

DS_SCRAPER   = DraftSharksScraper(timeout=30, delay=1.5)
PHASE        = "post_draft"
_failed      = []
_all_reviews = []

print("=" * 60)
try:
    rankings_df, rank_date = DS_SCRAPER.scrape()
except Exception as e:
    print(f"ERROR scraping DraftSharks: {e}")
    _failed.append("DraftSharks")
    rankings_df = None

if rankings_df is not None:
    try:
        _dim_rp, review = add_players_from_source(
            rankings_df, source_name="DraftSharks", draft_year=CFG.draft_year
        )
        if not review.empty:
            print(f"  -> {len(review)} players flagged for review")
            _all_reviews.append(review)
    except Exception as e:
        print(f"ERROR add_players: {e}")
        _failed.append("DraftSharks")

    try:
        ingest_ranking_source(
            rankings_df,
            source_name="DraftSharks",
            source_site=DraftSharksScraper.SOURCES["DraftSharks"]["source_site"],
            phase=PHASE,
            draft_year=CFG.draft_year,
            rank_date=rank_date,
        )
    except Exception as e:
        print(f"ERROR ingesting DraftSharks: {e}")
        _failed.append("DraftSharks")

if _all_reviews:
    new_review = pd.concat(_all_reviews, ignore_index=True)
    rp = DATA / "review" / "review_fuzzy_matches.csv"
    if rp.exists():
        existing = pd.read_csv(rp)
        combined = pd.concat([existing, new_review], ignore_index=True)
        # keep="first" preserves already-filled action values from prior review sessions
        combined.drop_duplicates(subset=["new_name", "source"], keep="first", inplace=True)
        n_added = len(combined) - len(existing)
        combined.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file updated: +{max(n_added, 0)} new rows ({len(combined)} total) -> {rp}")
    else:
        new_review.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file: {rp}  ({len(new_review)} rows)")
    print("Fill \"action\" column (\"match\" or \"new\"), run 09_apply_fuzzy_review, then re-run.")

print("\n" + "=" * 60)
if _failed:
    print(f"Failed: {_failed}")
else:
    print("DraftSharks completed.")

Scraped DraftSharks: 90 players | 2026 Dynasty Rookie Rankings: Top 100 | rank_date=2026-04-30
Source: DraftSharks
  Already in dim_rookie_prospect : 81
  Auto-matched (>=90)       : 3
  New prospects added             : 0
  Needs manual review             : 6
  -> 6 players flagged for review
WARN: 9 players pending review -- will be picked up after apply_review_decisions():
  De'Zhan Stribling
  Eli Raridon
  Jamarion Miller
  Emmanuel Henderson Jr.
  Cyrus Allen
  Casen Ryan
  CJ Williams
  Anthony Smith
  Da'Quan Wright
Ingested: 81 rows | source=DraftSharks | site=DraftSharks | phase=post_draft
fact_rookie_rankings total: 653

Review file updated: +6 new rows (67 total) -> data\review_fuzzy_matches.csv
Fill "action" column ("match" or "new"), run 09_apply_fuzzy_review, then re-run.

DraftSharks completed.


## 6 -- Validation

In [7]:
# Mini-validation: rows written by this source
import pandas as pd
from pathlib import Path
fr = pd.read_parquet(Path(CFG.data_dir) / "fact_rookie_rankings.parquet")
src_keys = list(SOURCES.keys())
sub = fr[fr["source_name"].isin(src_keys)]
print(f"Rows for this source: {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().to_string())
print()
print(f"fact_rookie_rankings total: {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key","source_name","phase","draft_year"]).sum()
print(f"Composite key duplicates: {dupes} (expected 0)")

Rows for this source: 81
source_name  phase     
DraftSharks  post_draft    81

fact_rookie_rankings total: 653 rows
Composite key duplicates: 0 (expected 0)
